In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.layers import Dense, Flatten, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

In [3]:
# Directories for dataset
TRAIN_DIR = "E:\\college\\grad project\\integrated\\training"
TEST_DIR = "E:\\college\\grad project\\integrated\\testing"
MODEL_SAVE_PATH = "saved_model/mri_classifier.keras"
CUSTOM_CNN_SAVE_PATH = "saved_model/custom_cnn_classifier.keras"

In [5]:
# Preprocessing function for training data
def preprocess_training_data(train_dir, img_size=(224, 224)):
    datagen = ImageDataGenerator(
        rescale=1.0/255.0,  # Normalize pixel values
        rotation_range=30,  # Increased rotation for augmentation
        width_shift_range=0.3,  # More aggressive augmentation
        height_shift_range=0.3,
        shear_range=0.3,
        zoom_range=0.3,
        horizontal_flip=True,
        validation_split=0.2  # Split training data for validation
    )

    train_generator = datagen.flow_from_directory(
        train_dir,
        target_size=img_size,S
        batch_size=32,
        class_mode='categorical',
        subset='training'
    )

    val_generator = datagen.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=32,
        class_mode='categorical',
        subset='validation'
    )

    return train_generator, val_generator

In [7]:
# Preprocessing function for testing data
def preprocess_testing_data(test_dir, img_size=(224, 224)):
    test_datagen = ImageDataGenerator(rescale=1.0/255.0)  # Only normalization for test data
    test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=img_size,
        batch_size=32,
        class_mode='categorical',
        shuffle=False
    )
    return test_generator

In [9]:
# Validate generator outputs
def validate_generator(generator):
    sample_batch = next(iter(generator))
    if len(sample_batch) != 2:
        raise ValueError(f"Generator yielded an element with length {len(sample_batch)}. Expected length is 2: (images, labels).")
    return True

In [11]:
# Load Data
train_gen, val_gen = preprocess_training_data(TRAIN_DIR)
test_gen = preprocess_testing_data(TEST_DIR)

Found 9345 images belonging to 4 classes.
Found 2333 images belonging to 4 classes.
Found 1705 images belonging to 4 classes.


In [13]:
# Validate Data Generators
validate_generator(train_gen)
validate_generator(val_gen)
validate_generator(test_gen)

True

In [15]:
# Calculate Class Weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

Class Weights: {0: 0.9578720787207872, 1: 0.9496951219512195, 2: 1.2023932063818836, 3: 0.9333799440671194}


In [17]:
# Model Definition for ResNet50V2
def create_resnet50v2_model(input_shape=(224, 224, 3), num_classes=4):
    base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=input_shape)
    for layer in base_model.layers[:20]:  # Freeze only the first 20 layers
        layer.trainable = False

    # Add custom classification layers
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [19]:
# Model Definition for Custom CNN
def create_custom_cnn_model(input_shape=(224, 224, 3), num_classes=4):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [21]:
# Create Models
resnet50_model = create_resnet50v2_model()
custom_cnn_model = create_custom_cnn_model()

C:\Users\BAB AL SAFA\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [23]:
# Training Callbacks
callbacks_resnet50 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),  # Learning rate scheduler
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True)
]

callbacks_custom_cnn = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),  # Learning rate scheduler
    ModelCheckpoint(CUSTOM_CNN_SAVE_PATH, monitor='val_accuracy', save_best_only=True)
]

In [29]:
# Train ResNet50V2 Model
resnet50_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    steps_per_epoch=len(train_gen),
    validation_steps=len(val_gen),
    class_weight=class_weights,
    callbacks=callbacks_resnet50
)

Epoch 1/25


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.

In [23]:
# Train Custom CNN Model
custom_cnn_model.fit(
    train_gen,
    validation_data=val_gen,  # Use validation data
    epochs=25,
    class_weight=class_weights,  # Add class weights
    callbacks=callbacks_custom_cnn
)

Epoch 1/25
293/293 ━━━━━━━━━━━━━━━━━━━━ 627s 2s/step - accuracy: 0.7640 - loss: 0.6646 - val_accuracy: 0.5988 - val_loss: 2.2382 - learning_rate: 0.0010
Epoch 2/25
293/293 ━━━━━━━━━━━━━━━━━━━━ 598s 2s/step - accuracy: 0.9009 - loss: 0.3042 - val_accuracy: 0.7510 - val_loss: 1.4687 - learning_rate: 0.0010
Epoch 3/25
293/293 ━━━━━━━━━━━━━━━━━━━━ 592s 2s/step - accuracy: 0.9091 - loss: 0.2665 - val_accuracy: 0.6845 - val_loss: 0.9741 - learning_rate: 0.0010
Epoch 4/25
293/293 ━━━━━━━━━━━━━━━━━━━━ 593s 2s/step - accuracy: 0.9295 - loss: 0.2004 - val_accuracy: 0.8534 - val_loss: 0.5569 - learning_rate: 0.0010
Epoch 5/25
293/293 ━━━━━━━━━━━━━━━━━━━━ 592s 2s/step - accuracy: 0.9362 - loss: 0.1931 - val_accuracy: 0.6112 - val_loss: 1.6393 - learning_rate: 0.0010
Epoch 6/25
293/293 ━━━━━━━━━━━━━━━━━━━━ 594s 2s/step - accuracy: 0.9270 - loss: 0.2107 - val_accuracy: 0.8453 - val_loss: 0.6325 - learning_rate: 0.0010
Epoch 7/25
293/293 ━━━━━━━━━━━━━━━━━━━━ 594s 2s/step - accuracy: 0.9543 - loss: 0.

In [25]:
# Save the models explicitly
resnet50_model.save(MODEL_SAVE_PATH)
custom_cnn_model.save(CUSTOM_CNN_SAVE_PATH)

In [41]:
# Load Models for Ensemble
def load_models():
    resnet50 = load_model(MODEL_SAVE_PATH)
    custom_cnn = load_model(CUSTOM_CNN_SAVE_PATH)
    return resnet50, custom_cnn

In [55]:
# Ensemble Prediction Function
def ensemble_predict(image_path, models):
    img = Image.open(image_path).resize((224, 224))
    img_array = np.expand_dims(np.array(img) / 255.0, axis=0)  # Normalize and batch

    # Get predictions from all models
    predictions = [model.predict(img_array) for model in models]
    averaged_predictions = np.mean(predictions, axis=0)  # Average the predictions

    class_names = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]
    return class_names[np.argmax(averaged_predictions)]

In [103]:
# Evaluate Ensemble Method
def evaluate_ensemble(test_dir, models):
    test_generator = preprocess_testing_data(test_dir)

    y_true = test_generator.classes
    y_pred = []

    for batch, _ in test_generator:
        for img in batch:
            img_array = np.expand_dims(img, axis=0)
            predictions = [model.predict(img_array) for model in models]
            averaged_predictions = np.mean(predictions, axis=0)
            y_pred.append(np.argmax(averaged_predictions))

    class_names = list(test_generator.class_indices.keys())

    print("\nClassification Report for Ensemble:\n")
    print(classification_report(y_true, y_pred, target_names=class_names))

    print("\nConfusion Matrix for Ensemble:\n")
    print(confusion_matrix(y_true, y_pred))

In [61]:
evaluate_ensemble(TEST_DIR, [trained_resnet50_model, trained_custom_cnn_model])

Found 1705 images belonging to 4 classes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━

In [63]:
# Evaluate Ensemble Method
def evaluate_ensemble(test_dir, models):
    test_generator = preprocess_testing_data(test_dir)

    y_true = test_generator.classes
    y_pred = []

    # Iterate through the test generator
    for i in range(len(test_generator)):
        batch_x, _ = test_generator[i]
        for img in batch_x:
            img_array = np.expand_dims(img, axis=0)
            predictions = [model.predict(img_array) for model in models]
            averaged_predictions = np.mean(predictions, axis=0)
            y_pred.append(np.argmax(averaged_predictions))

    # Ensure y_pred and y_true lengths match
    y_pred = y_pred[:len(y_true)]

    class_names = list(test_generator.class_indices.keys())

    # Print the evaluation metrics
    print("\nClassification Report for Ensemble:\n")
    print(classification_report(y_true, y_pred, target_names=class_names))

    print("\nConfusion Matrix for Ensemble:\n")
    print(confusion_matrix(y_true, y_pred))

In [65]:
# Upload and Predict Feature
def upload_and_predict(models):
    from tkinter import Tk, filedialog
    Tk().withdraw()  # Prevent Tkinter GUI from opening
    file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg *.jpeg *.png")])
    if file_path:
        result = ensemble_predict(file_path, models)
        print(f"Predicted Class: {result}")
    else:
        print("No file selected.")

In [67]:
# Self-Improvement Functionality
def retrain_model_with_new_data(new_data_dir, model_save_paths):
    new_train_gen, _ = preprocess_training_data(new_data_dir)
    models = load_models()

    for model, save_path in zip(models, model_save_paths):
        model.fit(
            new_train_gen,
            epochs=10,  # Fewer epochs for incremental training
            callbacks=[
                EarlyStopping(monitor='loss', patience=3, restore_best_weights=True),
                ReduceLROnPlateau(monitor='loss', factor=0.5, patience=2),  # Learning rate scheduler
                ModelCheckpoint(save_path, monitor='accuracy', save_best_only=True)
            ]
        )
        model.save(save_path)
        print(f"Model retrained and saved at {save_path}.")

In [69]:
# Example Usage (for testing):
trained_resnet50_model, trained_custom_cnn_model = load_models()

In [105]:
# Example ensemble prediction
result = ensemble_predict("E://college//grad project//integrated//testing//glioma_tumor//image(40).jpg", [trained_resnet50_model, trained_custom_cnn_model])
print(f"Ensemble Predicted Class: {result}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Ensemble Predicted Class: meningioma_tumor


In [107]:
from collections import Counter

train_class_counts = Counter(train_gen.classes)
val_class_counts = Counter(val_gen.classes)
print("Training Class Distribution:", train_class_counts)
print("Validation Class Distribution:", val_class_counts)

Training Class Distribution: Counter({3: 2503, 1: 2460, 0: 2439, 2: 1943})
Validation Class Distribution: Counter({3: 625, 1: 614, 0: 609, 2: 485})
